# 🚀 Training Local AI Gatekeeper (Scene Filter v2)
### Project: **ClipperVPS** - Affiliate Video Processing Pipeline

Notebook ini melatih model **MobileNetV3-Small** untuk klasifikasi biner frame video:
- **Class 0 (`valid_real`)**: Frame nyata produk (hands-on demo, penggunaan produk, unboxing, tekstur asli).
- **Class 1 (`rejected`)**: Frame kotor (kartun/2D animasi, intro bumper, outro sponsor, banner teks fullscreen, watermark tebal, slide statis).

Model yang dihasilkan diekspor ke **`scene_filter_v2.onnx`** (latency <10ms di CPU VPS) dan langsung kompatibel dengan `server/gatekeeper/service.py`.

In [ ]:
# 1. Setup & Cek Akselerasi GPU
import torch
import torchvision
import os
import sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"Device yang digunakan: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

!pip install -q onnx onnxruntime

In [ ]:
# 2. Upload Dataset (dataset_v2.zip dari collect_dataset.py)
from google.colab import files
import zipfile
import shutil

if not os.path.exists('dataset_v2.zip'):
    print("Silakan upload dataset_v2.zip hasil dari running 'python collect_dataset.py' di ClipperVPS:")
    uploaded = files.upload()

# Ekstrak dataset
if os.path.exists('dataset_v2.zip'):
    print("Mengekstrak dataset_v2.zip...")
    with zipfile.ZipFile('dataset_v2.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Ekstraksi selesai!")

# Verifikasi isi dataset
for split in ['train', 'val']:
    for cls in ['valid_real', 'rejected']:
        p = os.path.join('dataset', split, cls)
        count = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f"  [{split.upper()}] {cls}: {count} gambar")

In [ ]:
# 3. Data Augmentation & DataLoader
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('dataset/train', transform=train_transform)
val_dataset = datasets.ImageFolder('dataset/val', transform=val_transform)

print(f"Class mapping: {train_dataset.class_to_idx}")
assert train_dataset.class_to_idx == {'rejected': 1, 'valid_real': 0} or train_dataset.class_to_idx['valid_real'] == 0, \
    "Pastikan class_to_idx valid_real bernilai 0 dan rejected bernilai 1"

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total batch Train: {len(train_loader)}, Val: {len(val_loader)}")

In [ ]:
# 4. Definisi Model MobileNetV3-Small & Loss Function
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Load pre-trained MobileNetV3-Small
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

# Ganti output head classifier menjadi 2 kelas: [valid_real, rejected]
in_features = model.classifier[3].in_features
model.classifier[2] = nn.Dropout(p=0.3, inplace=True)
model.classifier[3] = nn.Linear(in_features, 2)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)

print(model.classifier)

In [ ]:
# 5. Training Loop (Fine-Tuning)
import time

EPOCHS = 15
best_val_acc = 0.0
best_model_weights = None

print("Memulai Training MobileNetV3-Small Scene Classifier...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Training
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
    scheduler.step()
    train_acc = train_correct / max(1, train_total)
    avg_train_loss = train_loss / max(1, train_total)
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    val_acc = val_correct / max(1, val_total)
    avg_val_loss = val_loss / max(1, val_total)
    duration = time.time() - t0
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({duration:.1f}s) | "
          f"Train Loss: {avg_train_loss:.4f}, Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {avg_val_loss:.4f}, Acc: {val_acc*100:.2f}%")
          
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = model.state_dict().copy()
        torch.save(best_model_weights, 'best_scene_filter.pth')
        print(f"  --> Model terbaik tersimpan (Val Acc: {best_val_acc*100:.2f}%)")

print(f"\nTraining Selesai! Best Validation Accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# 6. Evaluasi Metrik & Precision / Recall
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

target_names = ['valid_real (0)', 'rejected (1)']
print("=== CLASSIFICATION REPORT ===")
print(classification_report(all_labels, all_preds, target_names=target_names))
print("=== CONFUSION MATRIX ===")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
# 7. Export ke format ONNX (scene_filter_v2.onnx)
model.eval()
model = model.to('cpu')

dummy_input = torch.randn(1, 3, 224, 224, requires_grad=False)
onnx_output_path = "scene_filter_v2.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_output_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

file_size_mb = os.path.getsize(onnx_output_path) / (1024 * 1024)
print(f"Model ONNX berhasil diekspor: {onnx_output_path} ({file_size_mb:.2f} MB)")

In [ ]:
# 8. Verifikasi Benchmark Inference dengan ONNX Runtime di CPU
import onnxruntime as ort
import numpy as np
import time

session = ort.InferenceSession(onnx_output_path, providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
dummy_data = np.random.randn(1, 3, 224, 224).astype(np.float32)

# Warmup
for _ in range(5):
    _ = session.run(None, {input_name: dummy_data})

# Benchmark 50 iterasi
latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    _ = session.run(None, {input_name: dummy_data})
    latencies.append((time.perf_counter() - t0) * 1000)

avg_lat = np.mean(latencies)
p95_lat = np.percentile(latencies, 95)
print(f"Benchmark CPU Inference (ONNX Runtime):")
print(f"  Rata-rata: {avg_lat:.2f} ms per frame")
print(f"  P95:       {p95_lat:.2f} ms per frame")
if avg_lat < 15.0:
    print("  STATUS: AMAT CEPAT! Sangat ideal untuk CPU VPS ClipperVPS (<15ms)!")

In [ ]:
# 9. Download File scene_filter_v2.onnx
print("Mendownload scene_filter_v2.onnx ke komputer lokal Anda...")
files.download(onnx_output_path)
print("Setelah terdownload, copy file ini ke clipperVPS/server/gatekeeper/models/scene_filter_v2.onnx")